# 11. 거주동별 대표 통근부담 산출

이 노트북은 10번에서 생성한 `commute_routes_analysis_ready.csv`를 입력으로 사용한다.

10번 데이터는 **거주동–근무동 OD 한 개당 한 행**이며, 이번 11번에서는 이를 **거주 행정동 한 개당 한 행**으로 집계한다.

## 이번 노트북에서 계산하는 핵심 결과

- 대표 편도 통근시간
- 대표 편도 통근거리
- 대표 편도 교통비
- 월 통근시간
- 월 교통비
- 통근시간 기회비용
- 주요 출근 목적지
- API 경로 포함률
- 교통비 산출 포함률
- 내부 통근 비중
- 도보·환승 관련 보조지표

## 중요한 처리 원칙

시간과 교통비는 같은 방식으로 계산하지 않는다.

- **통근시간·거리**: 내부 통근을 포함한 모든 선택 OD에 값이 있으므로 기존 `최종_가중치`를 그대로 사용한다.
- **교통비**: 내부 통근은 실제 요금을 알 수 없어 결측이므로, 요금이 존재하는 OD 안에서 가중치를 다시 1로 맞춘다.
- 내부 통근 요금을 임의로 0원으로 두지 않는다. 같은 행정동 안에서도 버스나 지하철을 이용할 수 있기 때문이다.


In [1]:
# =========================================================
# 0. 라이브러리
# =========================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## 1. 분석 기준 설정

월 통근시간과 월 교통비는 편도값만으로 계산할 수 없으므로 다음 가정이 필요하다.

### 월평균 출근일 수

기본값은 `21일`로 둔다.

이는 법으로 정해진 고정값이 아니라 분석을 위한 가정이다. 실제 값은 분석연도의 평일·공휴일 기준이나 공식 근로일수 통계로 교체할 수 있다.
공휴일과 연차 등 실제 비근무일을 고려하고 통근 부담이 과대 추정되는 것을 방지하기 위해, 월 근무일수는 22일이 아닌 21일로 보수적으로 가정하였다.

### 시간가치

시간가치는 시간당 `10,320원`을 적용한다.

지역 간 비교에서는 모든 행정동에 같은 시간가치를 적용해야 통근시간 차이만 비용 차이로 반영된다. 행정동마다 서로 다른 임금을 적용하면 지역별 임금과 직업구성 차이가 함께 섞인다.

월평균 출근일 수는 21일, 시간가치는 10,320원을 프로젝트 공통 기준으로 적용한다.


In [2]:
# =========================================================
# 1. 분석 파라미터
# =========================================================

MONTHLY_WORKDAYS = 21.0
HOURLY_TIME_VALUE_WON = 10_320.0

# 주요 목적지 이름을 결과에 몇 개까지 표시할지
TOP_DESTINATION_COUNT = 5


print("월평균 출근일 수:", MONTHLY_WORKDAYS)
print("시간가치(원/시간):", f"{HOURLY_TIME_VALUE_WON:,.0f}")


월평균 출근일 수: 21.0
시간가치(원/시간): 10,320


## 2. 입력·출력 경로

입력 파일은 10번에서 만든 다음 파일이다.

```text
project_data/processed/commute_routes_analysis_ready.csv
```

최종 결과는 거주동 하나당 한 행으로 저장한다.

```text
project_data/processed/commute_burden_by_home_dong.csv
```


In [3]:
# =========================================================
# 2. 프로젝트 경로 설정
# =========================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    REPO_DIR = CURRENT_DIR.parent
elif (CURRENT_DIR / "notebooks").exists():
    REPO_DIR = CURRENT_DIR
else:
    raise FileNotFoundError(
        "프로젝트 저장소 위치를 찾지 못했습니다.\n"
        f"현재 위치: {CURRENT_DIR}\n"
        "저장소 루트 또는 notebooks 폴더에서 실행하세요."
    )

WORKSPACE_DIR = REPO_DIR.parent
DATA_DIR = WORKSPACE_DIR / "project_data"
PROCESSED_DIR = DATA_DIR / "processed"

INPUT_FILE = (
    PROCESSED_DIR
    / "commute_routes_analysis_ready.csv"
)

OUTPUT_FILE = (
    PROCESSED_DIR
    / "commute_burden_by_home_dong.csv"
)

print("입력:", INPUT_FILE)
print("출력:", OUTPUT_FILE)


입력: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_routes_analysis_ready.csv
출력: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_burden_by_home_dong.csv


## 3. 10번 결과 불러오기 및 필수 변수 검사

이번 노트북은 아래 변수를 기준으로 계산한다.

- `출근_이동량`: 해당 OD의 출근 이동량
- `최종_가중치`: 누적 80%로 선택된 목적지 안에서 다시 정규화한 가중치
- `분석용_편도시간_분`: 외부 통근은 API 전체 경로시간, 내부 통근은 원본 OD 평균시간
- `분석용_편도거리_km`: 외부 통근은 API 거리, 내부 통근은 원본 OD 평균거리
- `분석용_편도요금_원`: 외부 대중교통 요금과 순수 도보 0원, 내부 통근은 결측
- `내부통근여부`: 거주동과 근무동 코드가 같은지 여부

필수 변수가 없으면 뒤에서 잘못된 결과를 만드는 대신 즉시 중단한다.


In [4]:
# =========================================================
# 3. 데이터 불러오기 및 필수 변수 검사
# =========================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        "10번 최종 CSV를 찾지 못했습니다.\n"
        f"확인 경로: {INPUT_FILE}"
    )

od = pd.read_csv(
    INPUT_FILE,
    encoding="utf-8-sig",
    low_memory=False,
)

od.columns = (
    od.columns.astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

required_columns = [
    "OD_KEY",
    "거주동 코드",
    "거주동 이름",
    "근무동 코드",
    "근무동 이름",
    "출근_이동량",
    "거주동_전체_출근량",
    "목적지_출근비중",
    "누적_출근비중",
    "선택목적지_출근량합",
    "최종_가중치",
    "내부통근여부",
    "분석용_편도시간_분",
    "분석용_편도거리_km",
    "분석용_편도요금_원",
]

missing_columns = [
    column
    for column in required_columns
    if column not in od.columns
]

if missing_columns:
    raise KeyError(
        "11번 계산에 필요한 변수가 없습니다.\n"
        f"없는 변수: {missing_columns}\n"
        f"현재 변수: {od.columns.tolist()}"
    )

print("입력 행 수:", f"{len(od):,}")
print("고유 OD 수:", f"{od['OD_KEY'].nunique():,}")
print("거주동 수:", f"{od['거주동 코드'].nunique():,}")
display(od.head())


입력 행 수: 30,839
고유 OD 수: 30,839
거주동 수: 428


,OD_KEY,거주동 코드,거주동 이름,근무동 코드,근무동 이름,목적지_순위,출근_이동량,거주동_전체_출근량,목적지_출근비중,누적_출근비중,...,도보_구간수,기차_이용구간수,교통수단_순서,이용노선,추천경로수,전체추천경로수,최종경로유형,경로값_산출방식,경로정보존재여부,요금정보존재여부
0,11110515_11110530,11110515,청운효자동,11110530,사직동,1,55710.25,564412.04,0.098705,0.098705,...,2.0,0.0,WALK → BUS → WALK,지선:1711,4.0,4.0,대중교통,TMAP_최종경로,True,True
1,11110515_11110615,11110515,청운효자동,11110615,종로1.2.3.4가동,2,47373.52,564412.04,0.083934,0.182639,...,3.0,0.0,WALK → BUS → WALK → BUS → WALK,지선:1020 → 간선:272,10.0,10.0,대중교통,TMAP_최종경로,True,True
2,11110515_11110515,11110515,청운효자동,11110515,청운효자동,3,43228.92,564412.04,0.076591,0.259230,...,NaN,NaN,NaN,NaN,NaN,NaN,내부통근_원본OD,원본_OD_내부통근,True,False
3,11110515_11140550,11110515,청운효자동,11140550,명동,4,21721.17,564412.04,0.038485,0.297715,...,3.0,0.0,WALK → BUS → WALK → BUS → WALK,지선:7022 → 순환:TOUR11,10.0,10.0,대중교통,TMAP_최종경로,True,True
4,11110515_11140520,11110515,청운효자동,11140520,소공동,5,17247.38,564412.04,0.030558,0.328273,...,2.0,0.0,WALK → BUS → WALK,지선:1711,10.0,10.0,대중교통,TMAP_최종경로,True,True


## 4. 자료형 정리와 기본 품질검사

CSV를 읽으면 코드가 숫자로 해석되거나 불리언이 문자열로 읽힐 수 있다. 집계 전에 자료형을 명확히 통일한다.

또한 다음을 확인한다.

- `OD_KEY` 중복 여부
- 시간·거리 결측 여부
- 외부 통근 요금 결측 여부
- 거주동별 `최종_가중치` 합이 1인지 여부

가중치 합이 `0.99999993` 또는 `1.00000007`처럼 보일 수 있는데, 이는 소수를 컴퓨터에 저장하면서 생기는 부동소수점 오차다. 허용 오차 `1e-6` 안이면 정상으로 판단한다.


In [5]:
# =========================================================
# 4. 자료형 정리 및 기본 품질검사
# =========================================================

for column in ["거주동 코드", "근무동 코드"]:
    od[column] = (
        od[column]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

numeric_columns = [
    "출근_이동량",
    "거주동_전체_출근량",
    "목적지_출근비중",
    "누적_출근비중",
    "선택목적지_출근량합",
    "최종_가중치",
    "분석용_편도시간_분",
    "분석용_편도거리_km",
    "분석용_편도요금_원",
]

for column in numeric_columns:
    od[column] = pd.to_numeric(
        od[column],
        errors="coerce",
    )

if od["내부통근여부"].dtype != bool:
    od["내부통근여부"] = (
        od["내부통근여부"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
    )

# 매핑되지 않은 경우 코드 비교로 보완
internal_missing = od["내부통근여부"].isna()

od.loc[
    internal_missing,
    "내부통근여부",
] = (
    od.loc[internal_missing, "거주동 코드"]
    .eq(od.loc[internal_missing, "근무동 코드"])
)

od["내부통근여부"] = od["내부통근여부"].astype(bool)

if od["OD_KEY"].duplicated().any():
    raise ValueError("입력 데이터에 중복 OD_KEY가 있습니다.")

if od["분석용_편도시간_분"].isna().any():
    raise ValueError("편도시간에 결측이 있습니다.")

if od["분석용_편도거리_km"].isna().any():
    raise ValueError("편도거리에 결측이 있습니다.")

external_fee_missing = (
    (~od["내부통근여부"])
    & od["분석용_편도요금_원"].isna()
).sum()

if external_fee_missing > 0:
    raise ValueError(
        "외부 통근 요금에 결측이 남아 있습니다.\n"
        f"결측 OD 수: {external_fee_missing:,}"
    )

weight_sum = (
    od.groupby(
        ["거주동 코드", "거주동 이름"],
        observed=True,
    )["최종_가중치"]
    .sum()
)

if not np.allclose(
    weight_sum.to_numpy(dtype=float),
    1.0,
    atol=1e-6,
):
    raise ValueError(
        "일부 거주동의 최종 가중치 합이 1이 아닙니다."
    )

print("내부 통근 OD:", f"{od['내부통근여부'].sum():,}")
print("외부 통근 OD:", f"{(~od['내부통근여부']).sum():,}")
print("내부 통근 요금 결측:", f"{od.loc[od['내부통근여부'], '분석용_편도요금_원'].isna().sum():,}")
print("거주동별 가중치 합 범위:", weight_sum.min(), "~", weight_sum.max())
print("기본 품질검사 완료")


내부 통근 OD: 428
외부 통근 OD: 30,411
내부 통근 요금 결측: 428
거주동별 가중치 합 범위: 0.99999993 ~ 1.00000007
기본 품질검사 완료


## 5. 대표 편도 통근시간과 거리

거주동별 대표 통근시간은 선택된 주요 목적지의 시간을 출근 이동량 기반 가중치로 평균낸다.

\[
대표\ 편도시간
=
\sum(OD별\ 편도시간 \times 최종가중치)
\]

거리도 같은 방식으로 계산한다.

\[
대표\ 편도거리
=
\sum(OD별\ 편도거리 \times 최종가중치)
\]

내부 통근도 실제 시간이 0인 것이 아니므로 포함한다. 10번에서 내부 통근의 시간과 거리는 원본 OD 평균값으로 이미 대체했다.


In [6]:
# =========================================================
# 5. 대표 편도 통근시간·거리 계산
# =========================================================

od["가중_편도시간_분"] = (
    od["분석용_편도시간_분"]
    * od["최종_가중치"]
)

od["가중_편도거리_km"] = (
    od["분석용_편도거리_km"]
    * od["최종_가중치"]
)

time_distance_summary = (
    od.groupby(
        ["거주동 코드", "거주동 이름"],
        as_index=False,
        observed=True,
    )
    .agg(
        대표_편도통근시간_분=(
            "가중_편도시간_분",
            "sum",
        ),
        대표_편도통근거리_km=(
            "가중_편도거리_km",
            "sum",
        ),
    )
)

display(time_distance_summary.head())


,거주동 코드,거주동 이름,대표_편도통근시간_분,대표_편도통근거리_km
0,11110515,청운효자동,26.280906,6.048398
1,11110530,사직동,21.436403,3.625867
2,11110540,삼청동,25.678547,5.169288
3,11110550,부암동,32.549414,7.575307
4,11110560,평창동,33.311038,9.420747


## 6. 대표 편도 교통비

교통비에는 내부 통근 428건의 결측이 존재한다. 내부 통근을 0원으로 놓으면 내부 통근 비중이 높은 지역의 교통비가 과도하게 낮아진다.

따라서 요금이 존재하는 OD만 대상으로 가중치를 다시 계산한다.

\[
요금재조정가중치
=
\frac{해당\ OD의\ 최종가중치}
{해당\ 거주동에서\ 요금이\ 존재하는\ OD의\ 최종가중치\ 합}
\]

그 후 대표 교통비를 계산한다.

\[
대표\ 편도교통비
=
\sum(OD별\ 편도요금 \times 요금재조정가중치)
\]

이 값은 “모든 OD의 교통비를 완전히 관측한 평균”이 아니라, **요금이 확인된 OD를 이용한 추정값**이다. 따라서 교통비 산출 포함률을 함께 제시해야 한다.


In [7]:
# =========================================================
# 6. 요금 존재 OD 안에서 가중치 재조정
# =========================================================

od["요금정보존재여부_11"] = (
    od["분석용_편도요금_원"].notna()
)

fee_weight_sum = (
    od.loc[od["요금정보존재여부_11"]]
    .groupby(
        ["거주동 코드", "거주동 이름"],
        observed=True,
    )["최종_가중치"]
    .transform("sum")
)

od["요금_재조정가중치"] = np.nan

od.loc[
    od["요금정보존재여부_11"],
    "요금_재조정가중치",
] = (
    od.loc[
        od["요금정보존재여부_11"],
        "최종_가중치",
    ]
    / fee_weight_sum
)

od["가중_편도요금_원"] = (
    od["분석용_편도요금_원"]
    * od["요금_재조정가중치"]
)

fee_summary = (
    od.groupby(
        ["거주동 코드", "거주동 이름"],
        as_index=False,
        observed=True,
    )
    .agg(
        대표_편도교통비_원=(
            "가중_편도요금_원",
            "sum",
        ),
    )
)

# 모든 요금이 결측인 거주동은 sum 결과 0이 될 수 있으므로 다시 결측 처리
fee_available_by_home = (
    od.groupby(
        ["거주동 코드", "거주동 이름"],
        observed=True,
    )["요금정보존재여부_11"]
    .any()
    .rename("요금존재")
    .reset_index()
)

fee_summary = fee_summary.merge(
    fee_available_by_home,
    on=["거주동 코드", "거주동 이름"],
    how="left",
)

fee_summary.loc[
    ~fee_summary["요금존재"],
    "대표_편도교통비_원",
] = np.nan

fee_summary = fee_summary.drop(columns="요금존재")

display(fee_summary.head())


,거주동 코드,거주동 이름,대표_편도교통비_원
0,11110515,청운효자동,1539.668374
1,11110530,사직동,1533.863064
2,11110540,삼청동,1416.033793
3,11110550,부암동,1544.631742
4,11110560,평창동,1585.481146


## 7. 경로 포함률과 교통비 산출 포함률

두 포함률은 의미가 다르다.

### 경로 포함률

선택된 주요 목적지 가운데 시간·거리 경로값이 존재하는 이동량 비율이다.

10번 결과에서는 내부 통근도 원본 OD 시간·거리로 채웠으므로 현재는 대부분 100%가 된다.

\[
경로포함률
=
\frac{경로시간과\ 거리가\ 존재하는\ OD의\ 출근이동량}
{선택된\ 전체\ OD의\ 출근이동량}
\]

### 교통비 산출 포함률

요금이 존재하는 OD가 선택된 전체 이동량 중 얼마나 되는지를 나타낸다.

\[
교통비산출포함률
=
\frac{요금이\ 존재하는\ OD의\ 출근이동량}
{선택된\ 전체\ OD의\ 출근이동량}
\]

내부 통근 비중이 높을수록 교통비 포함률이 낮아질 수 있다.


In [8]:
# =========================================================
# 7. 경로·교통비 포함률 계산
# =========================================================

od["경로정보존재여부_11"] = (
    od["분석용_편도시간_분"].notna()
    & od["분석용_편도거리_km"].notna()
)

od["경로포함_이동량"] = np.where(
    od["경로정보존재여부_11"],
    od["출근_이동량"],
    0,
)

od["요금포함_이동량"] = np.where(
    od["요금정보존재여부_11"],
    od["출근_이동량"],
    0,
)

coverage_summary = (
    od.groupby(
        ["거주동 코드", "거주동 이름"],
        as_index=False,
        observed=True,
    )
    .agg(
        선택_OD_출근이동량=(
            "출근_이동량",
            "sum",
        ),
        경로포함_출근이동량=(
            "경로포함_이동량",
            "sum",
        ),
        요금포함_출근이동량=(
            "요금포함_이동량",
            "sum",
        ),
    )
)

coverage_summary["API_경로포함률"] = (
    coverage_summary["경로포함_출근이동량"]
    / coverage_summary["선택_OD_출근이동량"]
)

coverage_summary["교통비_산출포함률"] = (
    coverage_summary["요금포함_출근이동량"]
    / coverage_summary["선택_OD_출근이동량"]
)

display(coverage_summary.head())


,거주동 코드,거주동 이름,선택_OD_출근이동량,경로포함_출근이동량,요금포함_출근이동량,API_경로포함률,교통비_산출포함률
0,11110515,청운효자동,451971.51,451971.51,408742.59,1.0,0.904355
1,11110530,사직동,509677.13,509677.13,358773.75,1.0,0.703924
2,11110540,삼청동,85406.68,85406.68,75827.46,1.0,0.887840
3,11110550,부암동,362753.57,362753.57,334967.32,1.0,0.923402
4,11110560,평창동,563113.51,563113.51,513823.83,1.0,0.912469


## 8. 내부 통근 비중

내부 통근 비중은 한 거주동의 선택된 출근 이동량 중 거주동과 근무동이 같은 이동이 차지하는 비율이다.

\[
내부통근비중
=
\frac{내부통근\ 출근이동량}
{선택된\ 전체\ 출근이동량}
\]

이 값은 직주근접 정도를 설명하는 보조지표다. 다만 내부 통근이 높다고 해서 반드시 도보 통근이 많다는 뜻은 아니다.


In [9]:
# =========================================================
# 8. 내부 통근 비중 계산
# =========================================================

od["내부통근_출근이동량"] = np.where(
    od["내부통근여부"],
    od["출근_이동량"],
    0,
)

internal_summary = (
    od.groupby(
        ["거주동 코드", "거주동 이름"],
        as_index=False,
        observed=True,
    )
    .agg(
        내부통근_출근이동량=(
            "내부통근_출근이동량",
            "sum",
        ),
        선택_OD_출근이동량_내부계산=(
            "출근_이동량",
            "sum",
        ),
    )
)

internal_summary["내부통근비중"] = (
    internal_summary["내부통근_출근이동량"]
    / internal_summary["선택_OD_출근이동량_내부계산"]
)

internal_summary = internal_summary.drop(
    columns="선택_OD_출근이동량_내부계산"
)

display(internal_summary.head())


,거주동 코드,거주동 이름,내부통근_출근이동량,내부통근비중
0,11110515,청운효자동,43228.92,0.095645
1,11110530,사직동,150903.38,0.296076
2,11110540,삼청동,9579.22,0.112160
3,11110550,부암동,27786.25,0.076598
4,11110560,평창동,49289.68,0.087531


## 9. 주요 출근 목적지 목록과 누적 출근 비중

각 거주동에서 출근 이동량이 큰 목적지를 상위 `TOP_DESTINATION_COUNT`개까지 문자열로 정리한다.

이 변수는 대표값 계산에 직접 사용하지 않고, 해당 거주동의 출근 흐름이 어디로 향하는지를 사람이 쉽게 확인하기 위한 설명 변수다.

또한 선택된 목적지들이 거주동 전체 출근량의 몇 퍼센트를 설명하는지 계산한다.

\[
주요목적지\ 누적출근비중
=
\frac{선택목적지\ 출근이동량합}
{거주동\ 전체출근량}
\]

누적 80% 기준으로 선택했기 때문에 값은 대체로 0.8 이상이다.


In [10]:
# =========================================================
# 9. 주요 출근 목적지 및 선택 누적 비중
# =========================================================

def build_top_destinations(group: pd.DataFrame) -> str:
    top = (
        group.sort_values(
            ["출근_이동량", "근무동 코드"],
            ascending=[False, True],
        )
        .head(TOP_DESTINATION_COUNT)
    )

    return ", ".join(
        top["근무동 이름"]
        .astype("string")
        .dropna()
        .tolist()
    )


destination_list = (
    od.groupby(
        ["거주동 코드", "거주동 이름"],
        observed=True,
    )
    .apply(
        build_top_destinations,
        include_groups=False,
    )
    .rename("주요_출근목적지_목록")
    .reset_index()
)

destination_coverage = (
    od.groupby(
        ["거주동 코드", "거주동 이름"],
        as_index=False,
        observed=True,
    )
    .agg(
        거주동_전체_출근량=(
            "거주동_전체_출근량",
            "first",
        ),
        선택목적지_출근량합=(
            "출근_이동량",
            "sum",
        ),
    )
)

destination_coverage["주요목적지_누적출근비중"] = (
    destination_coverage["선택목적지_출근량합"]
    / destination_coverage["거주동_전체_출근량"]
)

display(destination_list.head())
display(destination_coverage.head())


,거주동 코드,거주동 이름,주요_출근목적지_목록
0,11110515,청운효자동,"사직동, 종로1.2.3.4가동, 청운효자동, 명동, 소공동"
1,11110530,사직동,"사직동, 종로1.2.3.4가동, 명동, 여의동, 소공동"
2,11110540,삼청동,"종로1.2.3.4가동, 삼청동, 청운효자동, 사직동, 가회동"
3,11110550,부암동,"종로1.2.3.4가동, 부암동, 사직동, 명동, 평창동"
4,11110560,평창동,"종로1.2.3.4가동, 평창동, 사직동, 부암동, 명동"


,거주동 코드,거주동 이름,거주동_전체_출근량,선택목적지_출근량합,주요목적지_누적출근비중
0,11110515,청운효자동,564412.04,451971.51,0.800783
1,11110530,사직동,636380.75,509677.13,0.800900
2,11110540,삼청동,106609.77,85406.68,0.801115
3,11110550,부암동,452322.26,362753.57,0.801980
4,11110560,평창동,703357.45,563113.51,0.800608


## 10. 도보·환승 보조지표

도보시간과 환승횟수는 대표 통근부담을 설명하는 보조지표로 사용한다.

- 대표 도보시간: 전체 편도 통근시간 중 실제 도보 구간의 평균
- 대표 환승횟수: 대중교통 경로의 평균 환승 횟수

내부 통근은 세부 교통수단 정보가 없으므로 도보시간과 환승횟수가 결측이다. 따라서 해당 값이 존재하는 OD 안에서 가중치를 재조정해 계산한다.

이 값들은 핵심 통근시간·교통비보다 자료 포함률이 낮을 수 있으므로 별도의 포함률도 함께 계산한다.


In [11]:
# =========================================================
# 10. 도보·환승 보조지표
# =========================================================

def weighted_metric_with_available_rows(
    data: pd.DataFrame,
    value_column: str,
    result_column: str,
    coverage_column: str,
) -> pd.DataFrame:
    temp = data[
        [
            "거주동 코드",
            "거주동 이름",
            "출근_이동량",
            "최종_가중치",
            value_column,
        ]
    ].copy()

    temp[value_column] = pd.to_numeric(
        temp[value_column],
        errors="coerce",
    )

    temp["값존재"] = temp[value_column].notna()

    available_weight_sum = (
        temp.loc[temp["값존재"]]
        .groupby(
            ["거주동 코드", "거주동 이름"],
            observed=True,
        )["최종_가중치"]
        .transform("sum")
    )

    temp["재조정가중치"] = np.nan
    temp.loc[
        temp["값존재"],
        "재조정가중치",
    ] = (
        temp.loc[
            temp["값존재"],
            "최종_가중치",
        ]
        / available_weight_sum
    )

    temp["가중값"] = (
        temp[value_column]
        * temp["재조정가중치"]
    )

    temp["포함이동량"] = np.where(
        temp["값존재"],
        temp["출근_이동량"],
        0,
    )

    result = (
        temp.groupby(
            ["거주동 코드", "거주동 이름"],
            as_index=False,
            observed=True,
        )
        .agg(
            **{
                result_column: ("가중값", "sum"),
                "_포함이동량": ("포함이동량", "sum"),
                "_전체이동량": ("출근_이동량", "sum"),
                "_값존재": ("값존재", "any"),
            }
        )
    )

    result.loc[
        ~result["_값존재"],
        result_column,
    ] = np.nan

    result[coverage_column] = (
        result["_포함이동량"]
        / result["_전체이동량"]
    )

    return result.drop(
        columns=[
            "_포함이동량",
            "_전체이동량",
            "_값존재",
        ]
    )


optional_summaries = []

if "총도보시간_분" in od.columns:
    walk_summary = weighted_metric_with_available_rows(
        od,
        value_column="총도보시간_분",
        result_column="대표_편도도보시간_분",
        coverage_column="도보시간_산출포함률",
    )
    optional_summaries.append(walk_summary)

if "환승횟수" in od.columns:
    transfer_summary = weighted_metric_with_available_rows(
        od,
        value_column="환승횟수",
        result_column="대표_편도환승횟수",
        coverage_column="환승횟수_산출포함률",
    )
    optional_summaries.append(transfer_summary)

print("생성된 보조지표 테이블 수:", len(optional_summaries))


생성된 보조지표 테이블 수: 2


## 11. 월 통근시간과 월 교통비

출근과 퇴근을 모두 포함하기 위해 편도값에 2를 곱한다.

\[
월통근시간
=
대표편도시간 \times 2 \times 월평균출근일수
\]

\[
월교통비
=
대표편도교통비 \times 2 \times 월평균출근일수
\]

월 통근시간은 먼저 분 단위로 계산하고, 해석 편의를 위해 시간 단위도 함께 저장한다.


## 12. 통근시간 기회비용

통근시간 기회비용은 실제 현금 지출이 아니다. 통근에 사용한 시간을 다른 활동에 쓰지 못한 손실을 금액으로 환산한 비현금성 비용이다.

\[
월통근시간기회비용
=
\frac{월통근시간(분)}{60}
\times 시간가치
\]

교통비와 통근시간 기회비용은 의미가 다르므로 최종 결과에서도 별도 변수로 유지한다.


In [12]:
# =========================================================
# 11~12. 결과 결합, 월 환산, 기회비용
# =========================================================

home_summary = time_distance_summary.copy()

tables_to_merge = [
    fee_summary,
    coverage_summary,
    internal_summary,
    destination_list,
    destination_coverage,
]

tables_to_merge.extend(optional_summaries)

for table in tables_to_merge:
    home_summary = home_summary.merge(
        table,
        on=["거주동 코드", "거주동 이름"],
        how="left",
        validate="one_to_one",
    )

# 월 환산
home_summary["월_통근시간_분"] = (
    home_summary["대표_편도통근시간_분"]
    * 2
    * MONTHLY_WORKDAYS
)

home_summary["월_통근시간_시간"] = (
    home_summary["월_통근시간_분"]
    / 60
)

home_summary["월_통근교통비_원"] = (
    home_summary["대표_편도교통비_원"]
    * 2
    * MONTHLY_WORKDAYS
)

# 시간 기회비용
home_summary["월_통근시간_기회비용_원"] = (
    home_summary["월_통근시간_시간"]
    * HOURLY_TIME_VALUE_WON
)

# 분석에 사용한 가정값도 결과에 기록
home_summary["월평균_출근일수_가정"] = MONTHLY_WORKDAYS
home_summary["시간가치_원_시간"] = HOURLY_TIME_VALUE_WON

display(home_summary.head())


,거주동 코드,거주동 이름,대표_편도통근시간_분,대표_편도통근거리_km,대표_편도교통비_원,선택_OD_출근이동량,경로포함_출근이동량,요금포함_출근이동량,API_경로포함률,교통비_산출포함률,...,대표_편도도보시간_분,도보시간_산출포함률,대표_편도환승횟수,환승횟수_산출포함률,월_통근시간_분,월_통근시간_시간,월_통근교통비_원,월_통근시간_기회비용_원,월평균_출근일수_가정,시간가치_원_시간
0,11110515,청운효자동,26.280906,6.048398,1539.668374,451971.51,451971.51,408742.59,1.0,0.904355,...,9.016529,0.904355,0.833019,0.904355,1103.798047,18.396634,64666.071710,189853.264090,21.0,10320.0
1,11110530,사직동,21.436403,3.625867,1533.863064,509677.13,509677.13,358773.75,1.0,0.703924,...,12.013150,0.703924,0.427788,0.703924,900.328906,15.005482,64422.248694,154856.571898,21.0,10320.0
2,11110540,삼청동,25.678547,5.169288,1416.033793,85406.68,85406.68,75827.46,1.0,0.887840,...,9.448853,0.887840,0.751041,0.887840,1078.498967,17.974983,59473.419298,185501.822371,21.0,10320.0
3,11110550,부암동,32.549414,7.575307,1544.631742,362753.57,362753.57,334967.32,1.0,0.923402,...,10.231002,0.923402,0.888348,0.923402,1367.075403,22.784590,64874.533183,235136.969381,21.0,10320.0
4,11110560,평창동,33.311038,9.420747,1585.481146,563113.51,563113.51,513823.83,1.0,0.912469,...,7.403933,0.912469,0.992191,0.912469,1399.063590,23.317726,66590.208119,240638.937434,21.0,10320.0


## 13. 최종 변수 정리

최종 파일은 거주동 하나당 한 행이다.

핵심 변수와 보조 변수를 구분해 배열한다.

- 식별 변수
- 대표 편도 통근값
- 월 환산값
- 시간 기회비용
- 목적지 설명 변수
- 포함률과 내부 통근 비중
- 도보·환승 보조지표
- 분석에 사용한 공통 가정값


In [13]:
# =========================================================
# 13. 최종 변수 정리
# =========================================================

final_columns = [
    "거주동 코드",
    "거주동 이름",
    "대표_편도통근시간_분",
    "대표_편도통근거리_km",
    "대표_편도교통비_원",
    "월_통근시간_분",
    "월_통근시간_시간",
    "월_통근교통비_원",
    "월_통근시간_기회비용_원",
    "주요_출근목적지_목록",
    "주요목적지_누적출근비중",
    "거주동_전체_출근량",
    "선택목적지_출근량합",
    "API_경로포함률",
    "교통비_산출포함률",
    "내부통근비중",
    "대표_편도도보시간_분",
    "도보시간_산출포함률",
    "대표_편도환승횟수",
    "환승횟수_산출포함률",
    "월평균_출근일수_가정",
    "시간가치_원_시간",
]

final_columns = [
    column
    for column in final_columns
    if column in home_summary.columns
]

commute_burden = (
    home_summary[final_columns]
    .sort_values(
        ["거주동 코드"],
        ascending=True,
    )
    .reset_index(drop=True)
)

display(commute_burden.head())
print("최종 거주동 수:", f"{len(commute_burden):,}")


,거주동 코드,거주동 이름,대표_편도통근시간_분,대표_편도통근거리_km,대표_편도교통비_원,월_통근시간_분,월_통근시간_시간,월_통근교통비_원,월_통근시간_기회비용_원,주요_출근목적지_목록,...,선택목적지_출근량합,API_경로포함률,교통비_산출포함률,내부통근비중,대표_편도도보시간_분,도보시간_산출포함률,대표_편도환승횟수,환승횟수_산출포함률,월평균_출근일수_가정,시간가치_원_시간
0,11110515,청운효자동,26.280906,6.048398,1539.668374,1103.798047,18.396634,64666.071710,189853.264090,"사직동, 종로1.2.3.4가동, 청운효자동, 명동, 소공동",...,451971.51,1.0,0.904355,0.095645,9.016529,0.904355,0.833019,0.904355,21.0,10320.0
1,11110530,사직동,21.436403,3.625867,1533.863064,900.328906,15.005482,64422.248694,154856.571898,"사직동, 종로1.2.3.4가동, 명동, 여의동, 소공동",...,509677.13,1.0,0.703924,0.296076,12.013150,0.703924,0.427788,0.703924,21.0,10320.0
2,11110540,삼청동,25.678547,5.169288,1416.033793,1078.498967,17.974983,59473.419298,185501.822371,"종로1.2.3.4가동, 삼청동, 청운효자동, 사직동, 가회동",...,85406.68,1.0,0.887840,0.112160,9.448853,0.887840,0.751041,0.887840,21.0,10320.0
3,11110550,부암동,32.549414,7.575307,1544.631742,1367.075403,22.784590,64874.533183,235136.969381,"종로1.2.3.4가동, 부암동, 사직동, 명동, 평창동",...,362753.57,1.0,0.923402,0.076598,10.231002,0.923402,0.888348,0.923402,21.0,10320.0
4,11110560,평창동,33.311038,9.420747,1585.481146,1399.063590,23.317726,66590.208119,240638.937434,"종로1.2.3.4가동, 평창동, 사직동, 부암동, 명동",...,563113.51,1.0,0.912469,0.087531,7.403933,0.912469,0.992191,0.912469,21.0,10320.0


최종 거주동 수: 428


## 14. 최종 품질검사

저장 전에 다음을 확인한다.

- 거주동 코드 중복이 없는가
- 대표 편도시간과 거리가 결측이 아닌가
- 대표 교통비 결측 거주동이 있는가
- 포함률이 0과 1 사이인가

대표 교통비가 결측인 거주동이 있다면 해당 거주동의 선택 OD가 전부 내부 통근이거나 요금이 없는 경우일 수 있다. 이때 0원으로 바꾸지 않고 별도 검토 대상으로 남긴다.


In [14]:
# =========================================================
# 14. 최종 품질검사
# =========================================================

if commute_burden["거주동 코드"].duplicated().any():
    raise ValueError("최종 결과에 중복 거주동 코드가 있습니다.")

if commute_burden["대표_편도통근시간_분"].isna().any():
    raise ValueError("대표 편도 통근시간에 결측이 있습니다.")

if commute_burden["대표_편도통근거리_km"].isna().any():
    raise ValueError("대표 편도 통근거리에 결측이 있습니다.")

for coverage_column in [
    "API_경로포함률",
    "교통비_산출포함률",
    "내부통근비중",
]:
    invalid = ~commute_burden[coverage_column].between(
        0,
        1,
        inclusive="both",
    )

    if invalid.any():
        raise ValueError(
            f"{coverage_column}에 0~1 범위 밖의 값이 있습니다."
        )

print(
    "대표 교통비 결측 거주동:",
    f"{commute_burden['대표_편도교통비_원'].isna().sum():,}",
)

print("\n[핵심 지표 요약]")
display(
    commute_burden[
        [
            "대표_편도통근시간_분",
            "대표_편도교통비_원",
            "월_통근교통비_원",
            "월_통근시간_기회비용_원",
            "교통비_산출포함률",
            "내부통근비중",
        ]
    ].describe()
)


대표 교통비 결측 거주동: 0

[핵심 지표 요약]


,대표_편도통근시간_분,대표_편도교통비_원,월_통근교통비_원,월_통근시간_기회비용_원,교통비_산출포함률,내부통근비중
count,428.000000,428.000000,428.000000,428.000000,428.000000,428.000000
mean,29.440075,1597.456242,67093.162152,212675.100101,0.907432,0.092568
std,4.696954,110.008310,4620.349026,33930.794743,0.068614,0.068614
min,15.461479,1398.005140,58716.215861,111693.721623,0.435638,0.016156
25%,26.402445,1540.915598,64718.455118,190731.266237,0.898727,0.054440
50%,29.215397,1575.312626,66163.130272,211052.028509,0.925630,0.074370
75%,32.519256,1626.262423,68303.021780,234919.105723,0.945560,0.101273
max,43.486049,2787.161705,117060.791627,314143.216517,0.983844,0.564362


## 15. 최종 CSV 저장

저장 파일:

```text
project_data/processed/commute_burden_by_home_dong.csv
```

이 파일은 이후 주거비 데이터와 결합해 총 주거·통근 부담을 계산하는 입력으로 사용할 수 있다.


In [15]:
# =========================================================
# 15. 최종 CSV 저장
# =========================================================

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

commute_burden.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료:", OUTPUT_FILE)
print("저장 행 수:", f"{len(commute_burden):,}")
print("저장 열 수:", f"{len(commute_burden.columns):,}")


저장 완료: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_burden_by_home_dong.csv
저장 행 수: 428
저장 열 수: 22


# 다음 분석 단계에 대한 메모

이 노트북은 현재 확보된 10번 경로 결과만으로 확정적으로 계산할 수 있는 항목을 산출한다.

PDF의 전체 최종 산출물 가운데 아래 변수는 별도 결과 파일이 준비된 뒤 추가 결합해야 한다.

- 출근·귀가 일치도
- 청년 목적지 코사인 유사도
- 청년 상위 5개 목적지 일치율
- 전체연령·청년 평균 이동시간 차이
- 루바인 통근권
- 통근권 집중도
- 통근권 내부 이동 비중

이 변수들을 근거 데이터 없이 임의로 만들지 않으며, 각각의 검증·네트워크 분석 결과가 생성된 뒤 `거주동 코드`를 기준으로 병합한다.
